In [27]:
import json
import requests
import openai

client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


In [28]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}


In [29]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다. 인기 영화를 추천하거나 보여줄 때 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 ID에 해당하는 영화의 상세 정보(제목, 줄거리, 평점 등)를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "정보를 가져올 영화의 고유 ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 영화 ID에 해당하는 영화의 출연진(배우) 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "출연진 정보를 가져올 영화의 고유 ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
]

In [30]:
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    choice = response.choices[0]

    # 모델이 도구 호출을 요청한 경우
    if choice.finish_reason == "tool_calls":
        tool_calls = choice.message.tool_calls
        messages.append(choice.message)  # assistant 메시지(tool_calls 포함) 추가

        for tool_call in tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"[도구 호출] {fn_name}({fn_args})")

            fn_result = FUNCTION_MAP[fn_name](**fn_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(fn_result, ensure_ascii=False),
            })

        # 도구 결과를 바탕으로 최종 응답 생성
        call_ai()
    else:
        message = choice.message.content
        messages.append({"role": "assistant", "content": message})
        print(f"AI: {message}")


In [33]:
while True:
    user_input = input("메시지를 입력하세요 (종료: q): ")
    if user_input.lower() == "q":
        break
    print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()

User: 지금 인기 있는 영화가 무엇인지 알려줘
[도구 호출] get_popular_movies({})
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**
   - **개봉일**: 2026-05-13
   - **장르**: 공포
   - **줄거리**: "One Wish Willow"의 신비로운 의미를 깨닫고 사랑의 감정을 얻으려는 낙천적인 주인공은 어둡고 음산한 대가를 치르게 됩니다.
   - **평점**: 7.9
   - ![Obsession](https://image.tmdb.org/t/p/w780/2G249T4Sgu8gXIZpaXWnxHYYNQV.jpg)

2. **Peddi**
   - **개봉일**: 2026-06-03
   - **장르**: 액션, 드라마
   - **줄거리**: 1980년대 안드라 프라데시의 시골에서 한 열정적인 마을 사람이 스포츠를 통해 지역 사회를 하나로 모아 강력한 적에게 맞섭니다.
   - **평점**: 6.3
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Lee Cronin's The Mummy**
   - **개봉일**: 2026-04-15
   - **장르**: 공포, 미스터리
   - **줄거리**: 기자의 어린딸이 8년 만에 돌아오지만, 기쁨의 재회는 악몽으로 변합니다.
   - **평점**: 8.1
   - ![Lee Cronin's The Mummy](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)

4. **The Mandalorian and Grogu**
   - **개봉일**: 2026-05-20
   - **장르**: 액션, 모험, SF
   - **줄거리**: 악의 제국이 무너진 후, 새로운 공화국이 전투를 위해 전설적인 만달로리안 현상금 사냥꾼과 그의 어린 제자를 찾습니다.
   - **평점**